# Statistical Significance Testing: Q-Learning vs Q-Learning Fuzzy vs HPA

This notebook performs statistical tests to determine whether there are significant differences in:
- **CPU Usage** (%)
- **Memory Usage** (%)
- **Replica Count**
- **Response Time** (ms)

**Methods:**
1. Descriptive Statistics
2. Normality Test (Shapiro-Wilk)
3. Kruskal-Wallis H Test (non-parametric one-way ANOVA)
4. Mann-Whitney U Test (pairwise comparisons)
5. Effect Size (Cohen's d & rank-biserial correlation)
6. Bonferroni-corrected p-values

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

In [ ]:
DATA_DIR = 'data-testing/11-02-2026'

df_ql = pd.read_csv(f'{DATA_DIR}/30-episodes-[1]/test_Q-LEARNING_20260212_173048.csv', parse_dates=['timestamp'])
df_qf = pd.read_csv(f'{DATA_DIR}/30-episodes-[1]/test_Q-LEARNING-FUZZY_20260212_173158.csv', parse_dates=['timestamp'])
df_hpa = pd.read_csv(f'{DATA_DIR}/30-episodes-[1]/metrics_20260212_181336.csv', parse_dates=['timestamp'])

# Label each method
df_ql['method'] = 'Q-Learning'
df_qf['method'] = 'Q-Learning Fuzzy'
df_hpa['method'] = 'HPA'

# Combine into one dataframe
df_all = pd.concat([df_ql, df_qf, df_hpa], ignore_index=True)

METRICS = ['cpu_usage', 'memory_usage', 'replica_state', 'response_time_ms']
METRIC_LABELS = {
    'cpu_usage': 'CPU Usage (%)',
    'memory_usage': 'Memory Usage (%)',
    'replica_state': 'Replica Count',
    'response_time_ms': 'Response Time (ms)'
}
METHODS = ['Q-Learning', 'Q-Learning Fuzzy', 'HPA']

print(f'Q-Learning     : {len(df_ql)} rows, NaN response_time: {df_ql["response_time_ms"].isna().sum()}')
print(f'Q-Learning Fuzzy: {len(df_qf)} rows, NaN response_time: {df_qf["response_time_ms"].isna().sum()}')
print(f'HPA             : {len(df_hpa)} rows, NaN response_time: {df_hpa["response_time_ms"].isna().sum()}')

## 1. Descriptive Statistics

In [ ]:
desc_stats = []
for method, df in [('Q-Learning', df_ql), ('Q-Learning Fuzzy', df_qf), ('HPA', df_hpa)]:
    for metric in METRICS:
        data = df[metric].dropna()
        desc_stats.append({
            'Method': method,
            'Metric': METRIC_LABELS[metric],
            'N': len(data),
            'Mean': data.mean(),
            'Std': data.std(),
            'Median': data.median(),
            'Min': data.min(),
            'Max': data.max(),
            'Q1 (25%)': data.quantile(0.25),
            'Q3 (75%)': data.quantile(0.75),
            'IQR': data.quantile(0.75) - data.quantile(0.25)
        })

df_desc = pd.DataFrame(desc_stats)
for metric_label in METRIC_LABELS.values():
    print(f'\n=== {metric_label} ===')
    display(df_desc[df_desc['Metric'] == metric_label].set_index(['Method']).drop(columns='Metric').round(4))

## 2. Distribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'Q-Learning': '#2196F3', 'Q-Learning Fuzzy': '#FF9800', 'HPA': '#4CAF50'}

for ax, metric in zip(axes.flatten(), METRICS):
    data_list = []
    labels = []
    for method in METHODS:
        d = df_all[df_all['method'] == method][metric].dropna()
        if len(d) > 0:
            data_list.append(d.values)
            labels.append(method)
    
    bp = ax.boxplot(data_list, labels=labels, patch_artist=True, widths=0.6)
    for patch, method in zip(bp['boxes'], labels):
        patch.set_facecolor(colors[method])
        patch.set_alpha(0.7)
    
    ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight='bold')
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.grid(True, alpha=0.3)

plt.suptitle('Distribution Comparison Across Methods', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Violin + strip plot for richer distribution view
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
palette = {'Q-Learning': '#2196F3', 'Q-Learning Fuzzy': '#FF9800', 'HPA': '#4CAF50'}

for ax, metric in zip(axes.flatten(), METRICS):
    plot_data = df_all[['method', metric]].dropna()
    sns.violinplot(data=plot_data, x='method', y=metric, palette=palette, ax=ax, inner='quartile', alpha=0.7)
    sns.stripplot(data=plot_data, x='method', y=metric, color='black', ax=ax, size=2, alpha=0.3)
    ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(METRIC_LABELS[metric])

plt.suptitle('Violin Plot: Distribution Shape Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 3. Normality Test (Shapiro-Wilk)

Before choosing between parametric (t-test/ANOVA) or non-parametric tests, we check if data is normally distributed.
- **H0**: Data is normally distributed
- **H1**: Data is NOT normally distributed
- If p < 0.05 → reject H0 → use non-parametric tests

In [ ]:
normality_results = []
for method, df in [('Q-Learning', df_ql), ('Q-Learning Fuzzy', df_qf), ('HPA', df_hpa)]:
    for metric in METRICS:
        data = df[metric].dropna()
        if len(data) >= 8:
            stat, p = stats.shapiro(data)
            normality_results.append({
                'Method': method,
                'Metric': METRIC_LABELS[metric],
                'Shapiro-Wilk Stat': stat,
                'p-value': p,
                'Normal (p>0.05)': 'Yes' if p > 0.05 else 'No'
            })

df_norm = pd.DataFrame(normality_results)
display(df_norm.style.applymap(
    lambda v: 'background-color: #c8e6c9' if v == 'Yes' else 'background-color: #ffcdd2' if v == 'No' else '',
    subset=['Normal (p>0.05)']
))

# Determine which test to use per metric
print('\n--- Test Selection ---')
for metric in METRICS:
    subset = df_norm[df_norm['Metric'] == METRIC_LABELS[metric]]
    all_normal = (subset['Normal (p>0.05)'] == 'Yes').all()
    test = 'One-Way ANOVA + t-test' if all_normal else 'Kruskal-Wallis + Mann-Whitney U'
    print(f'{METRIC_LABELS[metric]:25s} → All normal: {all_normal} → {test}')

## 4. Omnibus Test: Kruskal-Wallis H Test

Non-parametric alternative to one-way ANOVA. Tests whether at least one group differs significantly.
- **H0**: All groups come from the same distribution
- **H1**: At least one group differs
- If p < 0.05 → proceed with pairwise comparisons

In [ ]:
omnibus_results = []
for metric in METRICS:
    groups = []
    for method, df in [('Q-Learning', df_ql), ('Q-Learning Fuzzy', df_qf), ('HPA', df_hpa)]:
        d = df[metric].dropna()
        if len(d) > 0:
            groups.append(d.values)
    
    # Check normality to decide test
    all_normal = all(
        stats.shapiro(g)[1] > 0.05 for g in groups if len(g) >= 8
    )
    
    if all_normal and len(groups) == 3:
        stat, p = stats.f_oneway(*groups)
        test_name = 'One-Way ANOVA'
    else:
        stat, p = stats.kruskal(*groups)
        test_name = 'Kruskal-Wallis H'
    
    omnibus_results.append({
        'Metric': METRIC_LABELS[metric],
        'Test': test_name,
        'Statistic': stat,
        'p-value': p,
        'Significant (p<0.05)': 'Yes ***' if p < 0.001 else 'Yes **' if p < 0.01 else 'Yes *' if p < 0.05 else 'No'
    })

df_omnibus = pd.DataFrame(omnibus_results)
display(df_omnibus)

## 5. Pairwise Comparisons: Mann-Whitney U Test (with Bonferroni correction)

For each significant metric from the omnibus test, perform pairwise comparisons:
- Q-Learning vs Q-Learning Fuzzy
- Q-Learning vs HPA
- Q-Learning Fuzzy vs HPA

Bonferroni correction: multiply p-value by number of comparisons (3) to control for multiple testing.

In [ ]:
def cohens_d(x, y):
    """Calculate Cohen's d effect size."""
    nx, ny = len(x), len(y)
    pooled_std = np.sqrt(((nx - 1) * np.std(x, ddof=1)**2 + (ny - 1) * np.std(y, ddof=1)**2) / (nx + ny - 2))
    if pooled_std == 0:
        return 0.0
    return (np.mean(x) - np.mean(y)) / pooled_std

def effect_size_label(d):
    """Interpret Cohen's d."""
    d = abs(d)
    if d < 0.2: return 'Negligible'
    elif d < 0.5: return 'Small'
    elif d < 0.8: return 'Medium'
    else: return 'Large'

def rank_biserial(u_stat, n1, n2):
    """Calculate rank-biserial correlation from Mann-Whitney U."""
    return 1 - (2 * u_stat) / (n1 * n2)

method_data = {
    'Q-Learning': df_ql,
    'Q-Learning Fuzzy': df_qf,
    'HPA': df_hpa
}

pairs = list(combinations(METHODS, 2))
n_comparisons = len(pairs)  # 3

pairwise_results = []
for metric in METRICS:
    for m1, m2 in pairs:
        d1 = method_data[m1][metric].dropna().values
        d2 = method_data[m2][metric].dropna().values
        
        if len(d1) < 2 or len(d2) < 2:
            continue
        
        # Check normality
        normal_1 = stats.shapiro(d1)[1] > 0.05 if len(d1) >= 8 else False
        normal_2 = stats.shapiro(d2)[1] > 0.05 if len(d2) >= 8 else False
        
        if normal_1 and normal_2:
            # Levene's test for equal variances
            _, lev_p = stats.levene(d1, d2)
            equal_var = lev_p > 0.05
            stat, p = stats.ttest_ind(d1, d2, equal_var=equal_var)
            test_name = 't-test (equal var)' if equal_var else "Welch's t-test"
        else:
            stat, p = stats.mannwhitneyu(d1, d2, alternative='two-sided')
            test_name = 'Mann-Whitney U'
        
        p_bonf = min(p * n_comparisons, 1.0)
        d_effect = cohens_d(d1, d2)
        
        sig = ''
        if p_bonf < 0.001: sig = '***'
        elif p_bonf < 0.01: sig = '**'
        elif p_bonf < 0.05: sig = '*'
        else: sig = 'ns'
        
        pairwise_results.append({
            'Metric': METRIC_LABELS[metric],
            'Comparison': f'{m1} vs {m2}',
            'Test': test_name,
            'Statistic': round(stat, 4),
            'p-value (raw)': f'{p:.6f}',
            'p-value (Bonferroni)': f'{p_bonf:.6f}',
            'Significance': sig,
            'Cohen d': round(d_effect, 4),
            'Effect Size': effect_size_label(d_effect),
            'Mean A': round(np.mean(d1), 4),
            'Mean B': round(np.mean(d2), 4)
        })

df_pairwise = pd.DataFrame(pairwise_results)

for metric_label in METRIC_LABELS.values():
    print(f'\n{"="*80}')
    print(f' {metric_label}')
    print(f'{"="*80}')
    subset = df_pairwise[df_pairwise['Metric'] == metric_label].drop(columns='Metric')
    display(subset.reset_index(drop=True))

## 6. Summary Heatmap: Significance & Effect Size

In [ ]:
# Create significance heatmap
sig_map = {'***': 3, '**': 2, '*': 1, 'ns': 0}
effect_map = {'Large': 3, 'Medium': 2, 'Small': 1, 'Negligible': 0}

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Significance heatmap
pivot_sig = df_pairwise.pivot(index='Metric', columns='Comparison', values='Significance')
pivot_sig_num = pivot_sig.replace(sig_map)
sns.heatmap(pivot_sig_num, annot=pivot_sig.values, fmt='', cmap='RdYlGn_r', ax=axes[0],
            vmin=0, vmax=3, cbar_kws={'ticks': [0, 1, 2, 3], 'label': 'Significance Level'})
axes[0].set_title('Statistical Significance (Bonferroni-corrected)', fontweight='bold', fontsize=13)
axes[0].set_ylabel('')

# Effect size heatmap
pivot_eff = df_pairwise.pivot(index='Metric', columns='Comparison', values='Effect Size')
pivot_eff_num = pivot_eff.replace(effect_map)
annot_eff = df_pairwise.pivot(index='Metric', columns='Comparison', values='Cohen d')
annot_labels = pivot_eff.values.copy()
for i in range(annot_labels.shape[0]):
    for j in range(annot_labels.shape[1]):
        annot_labels[i, j] = f"{annot_eff.values[i, j]:.2f}\n({pivot_eff.values[i, j]})"

sns.heatmap(pivot_eff_num, annot=annot_labels, fmt='', cmap='YlOrRd', ax=axes[1],
            vmin=0, vmax=3, cbar_kws={'ticks': [0, 1, 2, 3], 'label': 'Effect Size'})
axes[1].set_title("Cohen's d Effect Size", fontweight='bold', fontsize=13)
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 7. Per-Metric Detailed Comparison Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
palette = {'Q-Learning': '#2196F3', 'Q-Learning Fuzzy': '#FF9800', 'HPA': '#4CAF50'}

for ax, metric in zip(axes.flatten(), METRICS):
    means = []
    stds = []
    medians = []
    for method in METHODS:
        d = method_data[method][metric].dropna()
        means.append(d.mean())
        stds.append(d.std())
        medians.append(d.median())
    
    x = np.arange(len(METHODS))
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=[palette[m] for m in METHODS],
                  alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.scatter(x, medians, color='red', zorder=5, s=50, marker='D', label='Median')
    
    # Add significance brackets
    subset = df_pairwise[df_pairwise['Metric'] == METRIC_LABELS[metric]]
    y_max = max(m + s for m, s in zip(means, stds)) * 1.1
    bracket_height = y_max * 0.08
    
    pair_positions = [(0, 1), (0, 2), (1, 2)]
    for idx, (_, row) in enumerate(subset.iterrows()):
        if row['Significance'] != 'ns':
            i, j = pair_positions[idx]
            y = y_max + bracket_height * idx * 1.5
            ax.plot([i, i, j, j], [y, y + bracket_height * 0.3, y + bracket_height * 0.3, y],
                    color='black', linewidth=1)
            ax.text((i + j) / 2, y + bracket_height * 0.35, row['Significance'],
                    ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.set_xticks(x)
    ax.set_xticklabels(METHODS)
    ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight='bold')
    ax.set_ylabel(METRIC_LABELS[metric])
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Mean ± Std Comparison with Significance Brackets', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Cumulative Distribution Function (CDF) Comparison

CDF shows the probability of observing a value ≤ x. Useful to compare the full distribution of each method.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'Q-Learning': '#2196F3', 'Q-Learning Fuzzy': '#FF9800', 'HPA': '#4CAF50'}

for ax, metric in zip(axes.flatten(), METRICS):
    for method in METHODS:
        data = method_data[method][metric].dropna().sort_values()
        cdf = np.arange(1, len(data) + 1) / len(data)
        ax.plot(data, cdf, label=method, color=colors[method], linewidth=2)
    
    ax.set_title(METRIC_LABELS[metric], fontsize=13, fontweight='bold')
    ax.set_xlabel(METRIC_LABELS[metric])
    ax.set_ylabel('Cumulative Probability')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Empirical CDF Comparison', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 9. Kolmogorov-Smirnov Test (Distribution Comparison)

Two-sample KS test: checks if two samples come from the same distribution.
Complements Mann-Whitney by testing the overall distribution shape, not just central tendency.

In [ ]:
ks_results = []
for metric in METRICS:
    for m1, m2 in pairs:
        d1 = method_data[m1][metric].dropna().values
        d2 = method_data[m2][metric].dropna().values
        
        if len(d1) < 2 or len(d2) < 2:
            continue
        
        stat, p = stats.ks_2samp(d1, d2)
        p_bonf = min(p * n_comparisons, 1.0)
        
        sig = '***' if p_bonf < 0.001 else '**' if p_bonf < 0.01 else '*' if p_bonf < 0.05 else 'ns'
        
        ks_results.append({
            'Metric': METRIC_LABELS[metric],
            'Comparison': f'{m1} vs {m2}',
            'KS Statistic': round(stat, 4),
            'p-value (Bonferroni)': f'{p_bonf:.6f}',
            'Significance': sig
        })

df_ks = pd.DataFrame(ks_results)
for metric_label in METRIC_LABELS.values():
    print(f'\n=== {metric_label} ===')
    display(df_ks[df_ks['Metric'] == metric_label].drop(columns='Metric').reset_index(drop=True))

## 10. Summary & Interpretation

In [ ]:
print('=' * 90)
print(' STATISTICAL ANALYSIS SUMMARY')
print('=' * 90)

for metric in METRICS:
    label = METRIC_LABELS[metric]
    print(f'\n--- {label} ---')
    
    # Omnibus result
    omni = df_omnibus[df_omnibus['Metric'] == label].iloc[0]
    print(f'  Omnibus test ({omni["Test"]}): p = {omni["p-value"]:.6f} → {omni["Significant (p<0.05)"]}')
    
    # Pairwise results
    pw = df_pairwise[df_pairwise['Metric'] == label]
    for _, row in pw.iterrows():
        direction = '>' if float(row['Mean A']) > float(row['Mean B']) else '<'
        print(f'  {row["Comparison"]:35s} | p(Bonf) = {row["p-value (Bonferroni)"]:>10s} '
              f'| {row["Significance"]:>3s} | d = {row["Cohen d"]:>7.3f} ({row["Effect Size"]:>10s}) '
              f'| {row["Mean A"]:>10.4f} {direction} {row["Mean B"]:>10.4f}')

# Overall winner analysis
print(f'\n{"="*90}')
print(' METHOD RANKING BY METRIC (lower is better for CPU, Memory, Response Time)')
print('=' * 90)
for metric in METRICS:
    label = METRIC_LABELS[metric]
    ranking = []
    for method in METHODS:
        mean_val = method_data[method][metric].dropna().mean()
        ranking.append((method, mean_val))
    
    # For replica_state, lower might be more efficient (less resource usage) or context-dependent
    ranking.sort(key=lambda x: x[1])
    
    print(f'\n  {label}:')
    for rank, (method, val) in enumerate(ranking, 1):
        print(f'    {rank}. {method:25s} → Mean = {val:.4f}')